# Analysis
Loads, cleans, and analyzes the healthcare appointment dataset.

Computed results are pickled to `analysis_results.pkl` at the end, so `02_visualization.ipynb` can pick them up without recomputing anything.

In [131]:
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')
from utils.filters import filter_appointments

In [132]:
# ── Load Data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('../data/appointments.csv', parse_dates=['appointment_date'])
df.rename(columns={'specialty': 'doctor_specialty'}, inplace=True)

print("=== Dataset Overview ===")
print(f"Total Records : {len(df)}")
print(f"Date Range    : {df['appointment_date'].min().date()} → {df['appointment_date'].max().date()}")
print(f"Divisions     : {df['division'].nunique()}")
print(f"Specialties   : {df['doctor_specialty'].nunique()}\n")

print("=== Column Data Types ===")
print(df.dtypes)

print("=== First 3 Rows ===")
df.head(3)

=== Dataset Overview ===
Total Records : 500
Date Range    : 2023-01-02 → 2024-12-26
Divisions     : 8
Specialties   : 6

=== Column Data Types ===
patient_age                      int64
patient_gender                     str
division                           str
doctor_specialty                   str
appointment_status                 str
consultation_fee_bdt             int64
wait_days                        int64
appointment_date        datetime64[us]
dtype: object
=== First 3 Rows ===


,patient_age,patient_gender,division,doctor_specialty,appointment_status,consultation_fee_bdt,wait_days,appointment_date
0,56,Female,Chattogram,Gynecologist,Completed,914,4,2023-11-19
1,19,Male,Barishal,Cardiologist,Completed,1546,6,2024-06-07
2,76,Female,Dhaka,Pediatrician,Completed,725,1,2023-02-27


In [133]:
# ── Basic Cleaning ─────────────────────────────────────────────────────────────
print("=== Missing Values ===")
print(df.isnull().sum())

=== Missing Values ===
patient_age             0
patient_gender          0
division                0
doctor_specialty        0
appointment_status      0
consultation_fee_bdt    0
wait_days               0
appointment_date        0
dtype: int64


In [134]:
# Age group segmentation
bins = [0, 17, 35, 55, 100]
labels = ['Child (0-17)', 'Young Adult (18-35)', 'Middle Age (36-55)', 'Senior (56+)']
df['age_group'] = pd.cut(df['patient_age'], bins=bins, labels=labels)

print("=== First 3 Rows after adding Age Groups column ===")
df.head(3)

=== First 3 Rows after adding Age Groups column ===


,patient_age,patient_gender,division,doctor_specialty,appointment_status,consultation_fee_bdt,wait_days,appointment_date,age_group
0,56,Female,Chattogram,Gynecologist,Completed,914,4,2023-11-19,Senior (56+)
1,19,Male,Barishal,Cardiologist,Completed,1546,6,2024-06-07,Young Adult (18-35)
2,76,Female,Dhaka,Pediatrician,Completed,725,1,2023-02-27,Senior (56+)


In [135]:
# Extract month name for trend analysis
df['month'] = df['appointment_date'].dt.month
df['year'] = df['appointment_date'].dt.year
df['month_name'] = df['appointment_date'].dt.strftime('%b')

print("=== First 3 Rows after adding Month Year columns ===")
df.head(3)

=== First 3 Rows after adding Month Year columns ===


,patient_age,patient_gender,division,doctor_specialty,appointment_status,consultation_fee_bdt,wait_days,appointment_date,age_group,month,year,month_name
0,56,Female,Chattogram,Gynecologist,Completed,914,4,2023-11-19,Senior (56+),11,2023,Nov
1,19,Male,Barishal,Cardiologist,Completed,1546,6,2024-06-07,Young Adult (18-35),6,2024,Jun
2,76,Female,Dhaka,Pediatrician,Completed,725,1,2023-02-27,Senior (56+),2,2023,Feb


### Correlation Analysis 

In [136]:
corr_matrix = df[['patient_age', 'consultation_fee_bdt', 'wait_days', 'month', 'year']].corr()
display(corr_matrix)

,patient_age,consultation_fee_bdt,wait_days,month,year
patient_age,1.000000,-0.058387,-0.000747,0.042031,-0.046246
consultation_fee_bdt,-0.058387,1.000000,-0.047631,-0.000729,0.102119
wait_days,-0.000747,-0.047631,1.000000,-0.001218,0.050077
month,0.042031,-0.000729,-0.001218,1.000000,0.013331
year,-0.046246,0.102119,0.050077,0.013331,1.000000


### Dynamic Filter Configuration
Set the division / specialty to drill into below. Every analysis cell and the summary at the end read from `filtered_df`, so changing these two lines re-scopes the whole notebook (and, once saved, the visualization notebook too).

In [137]:
# ── Dynamic Filter Configuration ────────────────────────────────────────────────
# Leave a value as None to include everything for that field.
FILTER_DIVISION = None        # e.g. "Dhaka", "Chattogram", or None for all divisions
FILTER_SPECIALTY = "Cardiologist"        # e.g. "Cardiologist", or None for all specialties

filtered_df = filter_appointments(df, division=FILTER_DIVISION, specialty=FILTER_SPECIALTY)

filter_label = " — ".join(p for p in [FILTER_DIVISION, FILTER_SPECIALTY] if p) or "All Divisions, All Specialties"

print(f"=== Active Filter: {filter_label} ===")
print(f"Filtered records: {len(filtered_df)} / {len(df)}")


=== Active Filter: Cardiologist ===
Filtered records: 85 / 500


### Analyses 1:
Appointment Status Breakdown by specialty of each division

In [138]:
status_percentage = (
    filtered_df.groupby(['division', 'doctor_specialty'])['appointment_status']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename('percentage')
)

status_count = (
    filtered_df.groupby(['division', 'doctor_specialty'])['appointment_status']
    .value_counts()
    .rename('count')
)

status_stats = (
    pd.concat([status_count, status_percentage], axis=1)
    .reset_index(['division', 'doctor_specialty', 'appointment_status'])
)

# Flat, chart-ready version — collapses the division/specialty breakdown above
# into overall counts for whatever is currently in filtered_df.
status_counts_vis = filtered_df['appointment_status'].value_counts()

print(f"=== Appointment Status ({filter_label}) ===")
display(status_stats)

=== Appointment Status (Cardiologist) ===


,division,doctor_specialty,appointment_status,count,percentage
0,Barishal,Cardiologist,Completed,3,60.0
1,Barishal,Cardiologist,Cancelled,1,20.0
2,Barishal,Cardiologist,No-show,1,20.0
3,Chattogram,Cardiologist,Completed,11,78.6
4,Chattogram,Cardiologist,Cancelled,2,14.3
5,Chattogram,Cardiologist,No-show,1,7.1
6,Dhaka,Cardiologist,Completed,19,67.9
7,Dhaka,Cardiologist,Cancelled,5,17.9
8,Dhaka,Cardiologist,No-show,4,14.3
9,Khulna,Cardiologist,Completed,8,66.7


### Analysis 2: 
No-show Rate only by Division 

In [139]:
noshow_by_division = (
    df.assign(is_noshow=(df['appointment_status'] == 'No-show'))
      .groupby('division')['is_noshow']
      .mean()
      .mul(100)
      .round(1)
      .reset_index(name='noshow_rate_pct')
      .sort_values('noshow_rate_pct', ascending=False)
)
noshow_by_division.columns = ['division', 'noshow_rate_pct']
print("=== No-show Rate by Division ===")
print(noshow_by_division.to_string(index=False))

=== No-show Rate by Division ===
  division  noshow_rate_pct
     Dhaka             14.6
Chattogram             13.4
  Rajshahi             11.7
    Sylhet              9.3
Mymensingh              8.0
    Khulna              6.7
  Barishal              4.5
   Rangpur              0.0


### Analysis 3: 
Analyses of Wait Days by Specialty of each division

In [140]:
avg_wait = (
    filtered_df.groupby(['division', 'doctor_specialty'])['wait_days']
    .agg(
        avg_wait_days='mean',
        median_wait_days='median',
        count='count',
        min_wait_days='min',
        max_wait_days='max',
        standard_deviation='std'
    )
    .round({'avg_wait_days': 1, 'median_wait_days': 1, 'standard_deviation': 1})
    .sort_values(by='avg_wait_days', ascending=False)
    .reset_index()
)

# Flat, chart-ready version — average wait purely by specialty, OR division,
# for whatever is currently in filtered_df.
group_by = 'division' if (FILTER_SPECIALTY and not FILTER_DIVISION) else 'doctor_specialty'
heading_label = 'Division' if group_by == 'division' else 'Specialty'

avg_wait_days_vis = (
    filtered_df.groupby(group_by)['wait_days']
    .mean()
    .round(1)
    .reset_index(name='avg_wait_days')
    .rename(columns={group_by: 'group'})
    .sort_values('avg_wait_days', ascending=False)
)

display(avg_wait_days_vis)

specialty_demand_vis = (
    filtered_df[group_by]
    .value_counts()
    .rename_axis('group')
    .reset_index(name='appointments_count')
)

avg_wait.dropna(subset=['standard_deviation'], inplace=True)
print(f"=== Wait Days by {heading_label} ({filter_label}) ===")
display(avg_wait)

,group,avg_wait_days
1,Chattogram,11.5
4,Mymensingh,11.4
7,Sylhet,10.6
3,Khulna,8.7
2,Dhaka,8.6
0,Barishal,7.6
5,Rajshahi,7.4
6,Rangpur,4.7


=== Wait Days by Division (Cardiologist) ===


,division,doctor_specialty,avg_wait_days,median_wait_days,count,min_wait_days,max_wait_days,standard_deviation
0,Chattogram,Cardiologist,11.5,12.0,14,2,20,6.9
1,Mymensingh,Cardiologist,11.4,12.0,7,2,18,5.5
2,Sylhet,Cardiologist,10.6,10.0,5,3,20,6.7
3,Khulna,Cardiologist,8.7,9.5,12,1,17,5.1
4,Dhaka,Cardiologist,8.6,8.0,28,1,19,5.7
5,Barishal,Cardiologist,7.6,8.0,5,1,14,4.7
6,Rajshahi,Cardiologist,7.4,7.0,8,1,16,5.2
7,Rangpur,Cardiologist,4.7,3.5,6,2,13,4.2


### Analysis 4: 
Age Group Distribution 

In [141]:
age_order = [
    'Child (0-17)',
    'Young Adult (18-35)',
    'Middle Age (36-55)',
    'Senior (56+)'
]

df['age_group'] = pd.Categorical(
    df['age_group'],
    categories=age_order,
    ordered=True
)

age_group_count = (
    filtered_df.groupby(['division', 'doctor_specialty'])['age_group']
    .value_counts()
    .rename('count')
)

age_group_percentage = (
    filtered_df.groupby(['division', 'doctor_specialty'])['age_group']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename('percentage')
)

age_group_stats = (
    pd.concat([age_group_count, age_group_percentage], axis=1)
    .reset_index(['age_group', 'division', 'doctor_specialty'])
)

# Flat, chart-ready version — age distribution of division/specialty.
age_dist_vis = (
    filtered_df['age_group']
    .value_counts()
    .reindex(age_order)
    .reset_index(name='count')
)

print("=== Patient Age Group Distribution ===")
display(age_group_stats)

=== Patient Age Group Distribution ===


,division,doctor_specialty,age_group,count,percentage
0,Barishal,Cardiologist,Young Adult (18-35),3,60.0
1,Barishal,Cardiologist,Middle Age (36-55),1,20.0
2,Barishal,Cardiologist,Child (0-17),1,20.0
3,Barishal,Cardiologist,Senior (56+),0,0.0
4,Chattogram,Cardiologist,Senior (56+),6,42.9
5,Chattogram,Cardiologist,Young Adult (18-35),4,28.6
6,Chattogram,Cardiologist,Child (0-17),3,21.4
7,Chattogram,Cardiologist,Middle Age (36-55),1,7.1
8,Dhaka,Cardiologist,Senior (56+),11,39.3
9,Dhaka,Cardiologist,Child (0-17),7,25.0


### Analysis 5: 
Monthly Appointment Trend 

In [142]:
monthly_trend = (
    df.groupby(['year', 'month', 'month_name'])
    .size()
    .reset_index(name='count')
    .sort_values(['year', 'month'])
)
print("=== Monthly Appointment Trend ===")
print(monthly_trend.to_string(index=False))

=== Monthly Appointment Trend ===
 year  month month_name  count
 2023      1        Jan     23
 2023      2        Feb     15
 2023      3        Mar     25
 2023      4        Apr     16
 2023      5        May     26
 2023      6        Jun     24
 2023      7        Jul     16
 2023      8        Aug     15
 2023      9        Sep     15
 2023     10        Oct     23
 2023     11        Nov     29
 2023     12        Dec     18
 2024      1        Jan     21
 2024      2        Feb     22
 2024      3        Mar     23
 2024      4        Apr     18
 2024      5        May     14
 2024      6        Jun     23
 2024      7        Jul     22
 2024      8        Aug     22
 2024      9        Sep     22
 2024     10        Oct     24
 2024     11        Nov     23
 2024     12        Dec     21


In [143]:
# ── Key Findings Summary ────────────────────
print(f"\n=== Key Findings — {filter_label} ===")

if len(filtered_df) == 0:
    print("No records match this filter — try loosening FILTER_DIVISION / FILTER_SPECIALTY.")
else:
    completion_rate = round((filtered_df['appointment_status'] == 'Completed').mean() * 100, 1)
    print(f"• Records in view                 : {len(filtered_df)} ({round(len(filtered_df) / len(df) * 100, 1)}% of all data)")
    print(f"• Completion rate                 : {completion_rate}%")

    if not noshow_by_division.empty:
        top_noshow = noshow_by_division.iloc[0]
        print(f"• Division with highest no-show   : {top_noshow['division']} ({top_noshow['noshow_rate_pct']}%)")

    if not specialty_demand_vis.empty:
        top_specialty = specialty_demand_vis.iloc[0]
        print(f"• Most visited specialty          : {top_specialty['group']} ({top_specialty['appointments_count']} appointments)")

    if not avg_wait_days_vis.empty:
        longest_wait = avg_wait_days_vis.iloc[0]
        print(f"• Longest average wait            : {longest_wait['group']} ({longest_wait['avg_wait_days']} days)")

    if not age_dist_vis.empty and age_dist_vis['count'].sum() > 0:
        top_age_group = age_dist_vis.loc[age_dist_vis['count'].idxmax()]
        print(f"• Largest patient group           : {top_age_group['age_group']} ({top_age_group['count']} patients)")



=== Key Findings — Cardiologist ===
• Records in view                 : 85 (17.0% of all data)
• Completion rate                 : 71.8%
• Division with highest no-show   : Dhaka (14.6%)
• Most visited specialty          : Dhaka (28 appointments)
• Longest average wait            : Chattogram (11.5 days)
• Largest patient group           : Senior (56+) (30 patients)


In [144]:
# ── Save Results for visualization.ipynb ───────────────────────────────────────
results = {
    'df': df,
    'filtered_df': filtered_df,
    'filter_division': FILTER_DIVISION,
    'filter_specialty': FILTER_SPECIALTY,
    'filter_label': filter_label,
    'status_stats': status_stats,
    'status_counts': status_counts_vis,
    'noshow_by_division': noshow_by_division,
    'avg_wait': avg_wait,
    'specialty_wait': avg_wait_days_vis,
    'specialty_demand': specialty_demand_vis,
    'header_label': heading_label,
    'age_group_stats': age_group_stats,
    'age_dist': age_dist_vis,
    'monthly_trend': monthly_trend,
}

with open('analysis_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print("Analysis results saved to analysis_results.pkl")


Analysis results saved to analysis_results.pkl
